# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
Exploration with `mlcroissant`

This notebook provides a practical step-by-step walkthrough for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library. All entities are referenced by their Croissant schema `@id`, ensuring dataset structure can be programmatically navigated and reused.

### Dataset Source
The dataset is provided as a Croissant schema:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and content using `mlcroissant`. This dataset is defined by a Croissant schema linked by a URL. We'll first inspect high-level information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant metadata file URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Cite as: {getattr(metadata, 'citeAs', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print("Authors (by @id):")
for author in getattr(metadata, 'author', []):
    print(f"- {author['@id']}")
print("\nFields containing possible sensitive information:")
for field in getattr(metadata, 'personalSensitiveInformation', []):
    print(f"- {field}")

## 2. Data Overview
We now review available record sets, their Croissant `@id`s, and field structure. This enables targeted exploration and downstream referencing.

**Note:** Use the Croissant `@id` for referencing record sets in further code blocks.

In [ ]:
# List all available record sets and the fields within them (all by @id)
# mlcroissant exposes record sets through the metadata object

print("Available Record Sets (by @id):\n")
record_sets = [r for r in getattr(metadata, 'recordSet', [])]
if not record_sets:
    print("No explicit record sets found as direct list (recordSet is empty).\n")
    print("Will enumerate all top-level record sets via dataset._record_sets (internal).")
    # Sometimes 'recordSet' is an empty list; use alternative API (internal but standard):
    record_sets = list(dataset._record_sets)  # This yields RecordSet objects
    for r in record_sets:
        print(f"- @id: {getattr(r, '@id', getattr(r, 'id_', None))}")
        print(f"  Name: {getattr(r, 'name', '')}")
        print(f"  Description: {getattr(r, 'description', '')}")
        # List fields by @id
        print("  Fields (@id):")
        for field in getattr(r, 'field', []):
            print(f"    - {field['@id']}")
        print()
else:
    for r in record_sets:
        print(f"- @id: {r['@id']}")
        print(f"  Name: {r.get('name', '')}")
        print(f"  Description: {r.get('description', '')}")
        print("  Fields (@id):")
        for field in r.get('field', []):
            print(f"    - {field['@id']}")
        print()

## 3. Data Extraction
We now extract the main survey data record set (and any additional record sets found) into pandas DataFrames for analysis.

Use the record set and field `@id` values from the overview above for precise extraction.

In [ ]:
# Collect all available record set @ids
record_set_objs = list(dataset._record_sets)
record_set_ids = [getattr(rs, '@id', getattr(rs, 'id_', None)) for rs in record_set_objs]
print("Extracting the following record sets (by @id):")
print(record_set_ids)

dataframes = {}
for rs in record_set_ids:
    # Each record set can be passed to records() for extraction
    print(f"\nLoading data for record set: {rs}")
    records = list(dataset.records(record_set=rs))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs] = df
        print(f"Columns in {rs}: {list(df.columns)}")
        display(df.head())
    else:
        print(f"No records found for record set {rs}.")
# For analysis, pick the main record set. We'll use the first found as an example:
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nChosen main record set for further EDA: {main_record_set_id}")
else:
    raise ValueError("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
We now conduct some basic EDA: filtering, normalization, and grouping. All field and column references use their Croissant `@id`s as shown above.

We will:
- Filter rows based on a selected numeric field (e.g., PHQ-9 or GAD-7 total score, using the field `@id`).
- Normalize this field.
- Group records by an important category (such as gender or education, using field `@id`).

_Edit the `numeric_field` and `group_field` variables as needed for your analysis. See displayed columns in the previous cell._

In [ ]:
# Select a main dataframe to analyze
df = dataframes[main_record_set_id]

# Choose numeric and group fields by their Croissant @id or column name
# Example: total PHQ-9 score field might be 'phq9_total_score' or similar @id
print("Available columns:\n", df.columns.tolist())

# Set these variables to match the actual column (field @id or the actual column name seen above)
numeric_field = 'phq9_total_score'  # <-- replace with actual field/column @id if different
group_field = 'gender'             # <-- replace with actual field/column @id if different

if numeric_field in df.columns:
    threshold = 5
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Records with {numeric_field} > {threshold}: {len(filtered_df)}")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Grouped analysis
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"\nMean {numeric_field} by {group_field} (> threshold):")
        display(grouped_df)
    else:
        print(f"Field '{group_field}' not found in columns. Skipping groupby.")
else:
    print(f"Numeric field '{numeric_field}' not found in columns! Please select a valid numeric field.")

## 5. Visualization
Let's visualize the distribution of the numeric field (e.g., PHQ-9 or GAD-7 total score) and compare across groups such as gender or education.

_Edit the field names below to match field @id or column names found above._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by category if applicable
    if group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
- The dataset provides comprehensive mental health, demographic, and survey data from Kilifi County, Kenya, in a machine-actionable Croissant structure.
- All extractions and analyses are referenced with the Croissant `@id` to ensure reproducibility and interoperability.
- The preliminary exploration reveals usable structure for modeling mental health scores by key factors (e.g., gender, education). You are encouraged to refine field `@id` selections per the columns shown, and to extend the EDA for your research questions.
